# Preprocess 0.25 deg ERA5 from glade

As hourly data, can get max from there as well. 

In [30]:
import xarray as xr
import xagg as xa
import pandas as pd
import numpy as np
import os
import glob
import re
from matplotlib import pyplot as plt
from cartopy import crs as ccrs
import cmocean
from tqdm.notebook import tqdm
from distributed import Client, worker_client


from funcs_support import get_params,get_filepaths,utility_save
from funcs_aux import _remove_chunk_encoding
dir_list = get_params()

In [2]:
# Start dask client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37729,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35579,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/42857/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:43169,


In [3]:
# File format: # /glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc/201001/e5.oper.an.sfc.128_167_2t.ll025sc.2010010100_2010013123.nc
params_fn = {'base':'/glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc/',
             'stem':'e5.oper.an.sfc.128_167_',
             'suffix':'.ll025sc.'}
# Variable to load
params_var = {'var':'2t','filevar':'VAR_2T','var_out':'tas',
              'long_name':'2 metre daily mean temperature',
              'short_name':'tas','units':'K',
              'extra_proc':{'max':{'var_out':'tasmax',
                                   'long_name':'2 metre daily max temperature',
                                   'short_name':'tasmax','units':'K'}}}


params_subset = {'lat':slice(-57,87),'lon':slice(-180,180)}

# MUST SET SCRATCH DIRECTORY HERE
output_dir = dir_list['raw']+'ERA5-025/'

process_as_mfdataset = True

In [4]:
# Get desired timesteps 
if 'oper.an' in params_fn['base']:
    #datelist = pd.date_range('1981-01-01','2024-12-31',freq='1MS')
    datelist = pd.date_range('1957-01-01','1980-12-31',freq='1MS')

    datestrs = [re.sub(r'\-','',str(datemon))[0:8]+'00_'+re.sub(r'\-','',str(datemon))[0:6]+str(datemon.daysinmonth)+'23'
            for datemon in datelist]
elif 'oper.fc' in params_fn['base']:
    # Forecast variables are stored in two files a month
    # (union sorts it)
    # There's a fn issue where the first half of the month files 
    # end on the 16th in the filename, but on the 15th (as expected)
    # in the file... 
    # Ending with 2023, there's something messed up with the later 2024 
    # files, there's an extra variable ('quantization_info'), not sure if
    # that's the culprit; but they don't open_mf with the others. 
    #datelist = pd.date_range('1981-01-01', '2024-01-31', freq='1MS').union(
    #                         pd.date_range('1980-12-01', '2024-01-31', freq='MS') + pd.Timedelta(days=15)
    #)
    datelist = pd.date_range('1957-01-01', '1981-01-31', freq='1MS').union(
                             pd.date_range('1956-12-01', '1981-01-31', freq='MS') + pd.Timedelta(days=15)
    )
    datestrs = [re.sub(r'\-','',str(datelist[n]))[0:8]+'06_'+re.sub(r'\-','',str(datelist[n+1]))[0:8]+'06'
            for n in range(len(datelist)-1)]



In [ ]:
if process_as_mfdataset:
    proc_list = [datestrs]
else:
    proc_list = datestrs


for proc_ in tqdm(proc_list):
    if type(proc_) == list:
        # Get filenames to open
        fns_open = [(params_fn['base']+datestr[0:6]+'/'+params_fn['stem']+
                   params_var['var']+params_fn['suffix']+datestr+'.nc')
                for datestr in proc_]

        timestr = datestrs[0][0:8]+'-'+datestrs[-1][11:19]

    else:
        datestr = proc_
        
        # Get filename
        fn_open = (params_fn['base']+datestr[0:6]+'/'+params_fn['stem']+
                   params_var['var']+params_fn['suffix']+datestr+'.nc')
    
        timestr = datestr[0:8]+'-'+datestr[11:19]

    params_var_set = {params_var['var_out']:{k:params_var[k] for k in params_var if k != 'extra_proc'},
                         **{params_var['extra_proc'][op]['var_out']:{**{k:pv for k,pv in params_var['extra_proc'][op].items()},
                                                                     'op':op}
                            for op in params_var['extra_proc']}}

    vars_out = [var for var in params_var_set]
    #vars_out = [params_var['var_out'],*[v['var_out'] for k,v in params_var['extra_proc'].items()]]
    output_fns = {var_out:output_dir+var_out+'_day_ERA5_historical_reanalysis_'+timestr+'.zarr'
                  for var_out in vars_out}

    file_exists = {var_out:os.path.exists(fn) for var_out,fn in output_fns.items()}

    if np.any([not ex for var_out,ex in file_exists.items()]):
        # Open file
        if process_as_mfdataset:
            ds = xr.open_mfdataset(fns_open,chunks='auto')
        else:
            ds = xr.open_dataset(fn_open,chunks='auto')
        if 'oper.fc' in params_fn['base']:
            # If forecast, stored as forecast_initial_time + forecast_hour
            ds = ds.stack(time=['forecast_initial_time','forecast_hour']).reset_index('time')
            ds['time'] = [t + pd.DateOffset(hours=int(h)) for t,h in zip(ds['forecast_initial_time'].values,
                                  ds['forecast_hour'].values)]
            ds['time'].attrs['DESCRIPTION'] = 'forecast_initial_time + forecast_hour'
            ds = ds.drop_vars(['forecast_initial_time','forecast_hour','utc_date'])
        
        # Standardize in name standards
        ds = xa.fix_ds(ds.rename({params_var['filevar']:params_var['var_out']})[[params_var['var_out']]])
        # Subset geographically
        ds = ds.sel(**params_subset)

        

        # If extra procs, do here:
        if len(params_var['extra_proc'])>0:
            extra_ds = []
            if 'max' in params_var['extra_proc']:
                extra_ds.append((ds.resample({'time':'1D'}).max().
                                 rename({params_var['var_out']:params_var['extra_proc']['max']['var_out']})))
            if 'min' in params_var['extra_proc']:
                extra_ds.append((ds.resample({'time':'1D'}).min().
                                 rename({params_var['var_out']:params_var['extra_proc']['min']['var_out']})))

            ds = xr.merge([ds,*extra_ds])

        # Get mean value
        ds = ds.resample({'time':'1D'}).mean()
        
        ds.attrs['SOURCE'] = 'preprocess_ERA5.ipynb'
        ds.attrs['DESCRIPTION'] = 'ERA5, from glade, standardized to file system standards'

        for var in ds:
            for attr in ['long_name','short_name','units']:
                ds[var].attrs[attr] = params_var_set[var][attr]
        
        for var_out in ds:
            if not file_exists[var_out]:
                utility_save(ds[[var_out]].chunk({'time':30,'lat':145,'lon':360}),output_fns[var_out])
            else:
                print(file_exists[var_out]+' exists, skipped!')
    else:
        print(', '.join([fn for var_out,fn in output_fns.items()])+' exist, skipped!')

In [5]:
fns = {var_out:glob.glob(output_dir+'/'+var_out+'_*.zarr')
       for var_out in ['tas','tasmax']}

In [8]:
subset_params = {'tas':[{},''],
                 'tasmax':[{},'']}

for var_out in ['tasmax']:#['tas','tasmax']:
    ds_out = xr.open_mfdataset(fns[var_out],
                      data_vars='minimal', coords='minimal', join='exact', compat='override')

    ds_out = ds_out.sel(**subset_params[var_out][0]).chunk({'time':30})
    # Since chunks are changed, have to remove the chunk encoding from the
    # original files, otherwise saving gets confused
    ds_out = _remove_chunk_encoding(ds_out)
    
    timestr = (re.sub(r'\-','',str(ds_out.time.min().values)[0:10])+'-'+
               re.sub(r'\-','',str(ds_out.time.max().values)[0:10]))
    fn_out = dir_list['raw']+'ERA5-025/'+var_out+'_day_ERA5-025_historical_reanalysis_'+timestr+subset_params[var_out][1]+'.zarr'
    
    utility_save(ds_out,fn_out,save_kwargs = {'zarr_format':2})

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/ERA5-025/tasmax_day_ERA5-025_historical_reanalysis_19810101-20241231.zarr saved!
